# [문제 1] Fashion MNIST 데이터 정규화를 위한 Mean과 Std. 값 찾기

In [14]:
import torch
from torch.utils.data import random_split, DataLoader
from torchvision import datasets, transforms
from wandb.util import download_file_into_memory

In [25]:
data_path = "."
f_mnist_train = datasets.FashionMNIST(data_path, train=True, download=True, transform=transforms.ToTensor())
f_mnist_train, f_mnist_validation = random_split(f_mnist_train, [55_000, 5_000])

images = torch.stack([img for img, _ in f_mnist_train], dim=0)

mean = images.mean()
std = images.std()

print(f"Mean: {mean.item():.4f}")
print(f"Std: {std.item():.4f}")

Mean: 0.2860
Std: 0.3530


# [문제 2] Fashion MNIST 데이터에 대하여 CNN 학습시키기

In [6]:
!pip install wandb

In [7]:
import wandb
wandb.login()

True

In [ ]:
!pip install torchinfo

In [1]:
import torch

if torch.cuda.is_available():
    print("GPU 사용 가능")
    print("GPU 개수:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i} 이름:", torch.cuda.get_device_name(i))
        print(f"GPU {i} CUDA 버전:", torch.version.cuda)
else:
    print("GPU 사용 불가, CPU 사용 중")


GPU 사용 불가, CPU 사용 중


In [ ]:
import os
from pathlib import Path
import torch
import wandb
from torch import nn

from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import transforms

BASE_PATH = str(Path(__file__).resolve().parent.parent.parent)  # BASE_PATH: /Users/yhhan/git/link_dl
print(BASE_PATH)

import sys

sys.path.append(BASE_PATH)

from _01_code._99_common_utils.utils import get_num_cpu_cores, is_linux, is_windows


def get_fashion_mnist_data():
    data_path = os.path.join(BASE_PATH, "_00_data", "j_fashion_mnist")

    f_mnist_train = datasets.FashionMNIST(data_path, train=True, download=True, transform=transforms.ToTensor())
    f_mnist_train, f_mnist_validation = random_split(f_mnist_train, [55_000, 5_000])

    print("Num Train Samples: ", len(f_mnist_train))
    print("Num Validation Samples: ", len(f_mnist_validation))
    print("Sample Data Shape: ", f_mnist_train[0][0].shape)  # torch.Size([1, 28, 28])
    print("Sample Data Target: ", f_mnist_train[0][1])  # 9

    num_data_loading_workers = get_num_cpu_cores()
    print("Number of Data Loading Workers:", num_data_loading_workers)

    train_data_loader = DataLoader(
        dataset=f_mnist_train, batch_size=wandb.config.batch_size, shuffle=True,
        pin_memory=True, num_workers=num_data_loading_workers
    )

    validation_data_loader = DataLoader(
        dataset=f_mnist_validation, batch_size=wandb.config.batch_size,
        pin_memory=True, num_workers=num_data_loading_workers
    )

    f_mnist_transforms = nn.Sequential(
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize(mean=0.2860, std=0.3530),
    )

    return train_data_loader, validation_data_loader, f_mnist_transforms


def get_fashion_mnist_test_data():
    data_path = os.path.join(BASE_PATH, "_00_data", "j_fashion_mnist")

    f_mnist_test_images = datasets.FashionMNIST(data_path, train=False, download=True)
    f_mnist_test = datasets.FashionMNIST(data_path, train=False, download=True, transform=transforms.ToTensor())

    print("Num Test Samples: ", len(f_mnist_test))
    print("Sample Shape: ", f_mnist_test[0][0].shape)  # torch.Size([1, 28, 28])

    test_data_loader = DataLoader(dataset=f_mnist_test, batch_size=len(f_mnist_test))

    f_mnist_transforms = nn.Sequential(
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize(mean=0.2860, std=0.3530),
    )

    return f_mnist_test_images, test_data_loader, f_mnist_transforms


if __name__ == "__main__":
    config = {'batch_size': 2048, }
    wandb.init(mode="disabled", config=config)

    train_data_loader, validation_data_loader, f_mnist_transforms = get_fashion_mnist_data()
    print()
    f_mnist_test_images, test_data_loader, f_mnist_transforms = get_fashion_mnist_test_data()


Namespace(wandb=False, batch_size=2048, epochs=500, learning_rate=0.001, validation_intervals=10, early_stop_patience=10, early_stop_delta=1e-05)
{'epochs': 500, 'batch_size': 2048, 'validation_intervals': 10, 'learning_rate': 0.001, 'early_stop_patience': 10, 'early_stop_delta': 1e-05, 'weight_decay': 0.0001}
Training on device cpu.
Num Train Samples:  55000
Num Validation Samples:  5000
Sample Data Shape:  torch.Size([1, 28, 28])
Sample Data Target:  1
Number of Data Loading Workers: 16


C:\Users\WJM\anaconda3\envs\link_dl\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[Epoch   1] T_loss: 0.85535, T_accuracy: 71.4582 | V_loss: 1.09089, V_accuracy: 62.5200 | Early stopping is stated! | T_time: 00:16:25, T_speed: 0.001
[Epoch  10] T_loss: 0.11880, T_accuracy: 95.9255 | V_loss: 0.47431, V_accuracy: 85.9000 | V_loss decreased (1.09089 --> 0.47431). Saving model... | T_time: 02:45:12, T_speed: 0.001
[Epoch  20] T_loss: 0.01616, T_accuracy: 99.6236 | V_loss: 0.49613, V_accuracy: 89.6000 | Early stopping counter: 1 out of 10 | T_time: 05:06:49, T_speed: 0.001
